## 1. Import & Setup
Load the dataset and immediately convert the timestamp column to a datetime object.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np

# Set visualization style for executive report
sns.set_theme(style="whitegrid", context="talk")

# Set up directories using pathlib
current_dir = Path().resolve()
project_root = current_dir.parent

data_dir = project_root / "data"
output_dir = project_root / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# Load the dataset
file_path = data_dir / "nsp_electricity_dataset.csv"
df = pd.read_csv(file_path)

# Convert the timestamp column to a datetime object immediately
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"Dataset loaded successfully. Shape: {df.shape}")

## 2. Structural Inspection
Display the dataframe's info(), head(), and check for any missing values across all columns.

In [ ]:
# Display the dataframe's info
print("--- Dataframe Info ---")
df.info()

# Display the first few rows
print("\n--- First 5 Rows ---")
display(df.head().T)

# Check for missing values across all columns
print("\n--- Missing Values Check ---")
missing_values = df.isnull().sum()
if missing_values.sum() == 0:
    print("No missing values found in the dataset.")
else:
    print(missing_values[missing_values > 0])

## 3. Statistical Verification
Identify any potential outliers where values are more than 3 standard deviations from the mean.

In [ ]:
display(df.describe().T)

## 4. Target Visualization
Plot a histogram of consumption_kwh using a log scale for the Y-axis to visualize demand spikes.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['consumption_kwh'], bins=100, kde=True, color='blue')
plt.title('Distribution of Electricity Consumption (kWh)', fontsize=14)
plt.xlabel('Consumption (kWh)')
plt.ylabel('Frequency (Log Scale)')
plt.yscale('log')
plt.savefig(output_dir / 'consumption_dist.png')
plt.show()

In [ ]:
# 1. Temperature vs Consumption (The Weather Factor)
plt.figure(figsize=(10, 6))
sns.regplot(data=df, x='temperature_c', y='consumption_kwh', 
            scatter_kws={'alpha':0.2, 's':10}, line_kws={'color':'red'})
plt.title('Relationship: Temperature vs. Electricity Consumption (Log-Log Scale)', fontsize=14)
plt.xlabel('Temperature (°C) [Log Scale]')
plt.ylabel('Consumption (kWh) [Log Scale]')
plt.xscale('log')
plt.yscale('log')
plt.savefig(output_dir / 'temp_vs_consumption.png')
plt.show()

## 5. Regional and Temporal Analysis

In [ ]:
pivot_df = df.pivot_table(values='consumption_kwh', index='region', columns='month', aggfunc='mean')
plt.figure(figsize=(12, 6))
sns.heatmap(pivot_df, cmap='YlOrRd', annot=True, fmt=".0f", linewidths=.5)
plt.title('Average Monthly Electricity Consumption by Region (kWh)', fontsize=16, fontweight='bold', pad=15)
plt.show()

## 6. Feature Engineering for Forecasting
Creating lags, Rolling Stats, and Cyclical encodings. Important for forecasting accuracy.

In [ ]:
# Sort for time-series splits
df = df.sort_values(by=['region', 'timestamp'])

# 6.1 Lag Features
df['consumption_kwh_lag_1h'] = df.groupby('region')['consumption_kwh'].shift(1)
df['consumption_kwh_lag_24h'] = df.groupby('region')['consumption_kwh'].shift(24)

# 6.2 Weather Features
df['hdd'] = (18 - df['temperature_c']).clip(lower=0)

# 6.3 Rolling Features (24h Window)
df['rolling_mean_24h'] = df.groupby('region')['consumption_kwh'].transform(lambda x: x.rolling(window=24, min_periods=1).mean())
df['rolling_std_24h'] = df.groupby('region')['consumption_kwh'].transform(lambda x: x.rolling(window=24, min_periods=1).std().fillna(0))

# 6.4 Cyclical Hour/Month Encoding
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# 6.5 Categorical Encoding
df_encoded = pd.get_dummies(df, columns=['region', 'customer_type'], drop_first=False)

# 6.6 Export to Output Folder
export_file = output_dir / 'engineered_features.csv'
df.dropna(inplace=True)
df.to_csv(export_file, index=False)
print(f"All feature engineering important for forecasting has been saved to: {export_file}")

## 1. Import & SetupLoad the dataset and immediately convert the timestamp column to a datetime object.====SEP====## 2. Structural InspectionDisplay the dataframe's info(), head(), and check for any missing values across all columns.====SEP====## 3. Statistical VerificationGenerate descriptive statistics for all numerical columns (specifically looking at the range of consumption_kwh and weather variables). Identify any potential outliers where values are more than 3 standard deviations from the mean.====SEP====## 4. Target VisualizationPlot a histogram of consumption_kwh using a log scale for the Y-axis to visualize the frequency of extreme demand spikes. Save the figure to the output directory.====SEP====### The "Long Tail" of Electricity ConsumptionLook at the Electricity Consumption histogram. You'll notice it's heavily skewed to the right.- The Insight: Most of the time, consumption stays in a lower, predictable range (the high peak on the left). However, the "tail" stretches far to the right, representing those rare but critical moments of extreme high demand (spikes).- Scientific Note: This is high Kurtosis. It tells us that our electrical grid must be built to handle these extreme "outlier" events, even if they don't happen often.====SEP====### The Weather PatternThe Temperature distribution shows a bell-like curve but with a wide spread.- The Insight: Unlike consumption, temperature is more "normally" distributed but covers a vast range (from deep freeze to summer heat).- Scientific Note: The spread (variance) here is key for energy forecasting. We can see that the grid operates in temperatures ranging from $-40^\circ C$ to over $+40^\circ C$, requiring highly resilient infrastructure.====SEP====### Balanced vs. Imbalanced DataData balance is crucial for training fair machine learning models.- Region (Balanced): The bar chart for Region shows five identical bars. This is a "Uniform Distribution." It means we have an equal amount of data for Halifax as we do for Cape Breton. This is perfect—it ensures our analysis isn't biased toward one city.- Anomaly Type (Imbalanced): Contrast that with the Anomaly Type chart (on a log scale). The "Normal" bar towers over the others.- Scientific Note: This is a classic Class Imbalance problem. Since anomalies like "Spikes" or "Power Outages" are so rare (less than 1%), we have to use specialized techniques to "teach" an AI how to recognize them.====SEP====### Who is using the power?The Customer Type pie chart gives us a snapshot of the grid's "clientele."- The Insight: Residential users are the primary drivers of demand ($43.5\%$).- Scientific Note: Because residential usage is often tied to human behavior (waking up, cooking dinner), we expect strong daily cycles in this dataset.====SEP====### Temperature vs. Usage (The "Heating" Effect)When we look at Temperature vs. Consumption, we see a clear trend.- The Simple Insight: As it gets colder (moving left on the graph), electricity usage goes up.- Why it matters: This suggests that many customers likely use electric heating. The red line shows the general "downward" slope—warmer weather generally means lower power bills for the region.- The Outliers: Notice the dots scattered way above the line? Those are the "spikes" we discussed earlier, occurring even in mild weather, likely due to industrial activity or grid anomalies.====SEP====### The Daily Heartbeat (Usage by Hour)This line chart shows the Average Consumption by Hour of the day.- The Simple Insight: People use the most power at 7:00 AM (waking up) and 6:00 PM (coming home/dinner).- Why it matters: Usage is lowest in the middle of the night (3:00 AM). The "Double Hump" pattern is the classic heartbeat of a residential-heavy grid. This is when the grid is under the most stress.====SEP====### Green Energy vs. PollutionWe compared Renewable Energy % against CO2 Emissions.- The Simple Insight: The dots trend downward to the right. This means when we use more wind/solar/hydro (Renewables), our carbon footprint (CO2) drops significantly.- Why it matters: It proves that our green energy initiatives are working. Notice the different colored dots—Commercial/Industrial users (green) often have a much higher footprint than Residential users (orange), even when green energy is high.====SEP====### Who uses the most power?The bar chart for Customer Type shows the average "size" of a typical power bill.- The Simple Insight: On average, a single Commercial/Industrial connection uses more power than a Residential one.- Why it matters: While there are more residents, the industrial sector is the "heavy hitter." If we need to save a lot of energy quickly (during a shortage), the biggest impact comes from working with industrial partners first.====SEP====## Summary for Decision Makers:1. Prepare for the Cold: Ensure the grid has extra capacity when temperatures drop below $0^\circ C$.2. Manage the Peak: Focus energy-saving programs on the 6:00 PM peak to reduce grid stress.3. Invest in Green: Increasing renewable energy by even $10\%$ has a massive, visible impact on lowering CO2 levels across all customer types.====SEP====### Pair Plots: The "Big Picture" ViewA Pair Plot is like a speed-dating session for your data. It plots every variable against every other variable simultaneously.- Simple Insight: Look at the diagonal row; it shows the "individual" distributions. Look at the scatter plots; they show how groups behave.- What we see: You can notice distinct clusters forming based on customer_type (the colors). For instance, Commercial users (orange) often cluster in higher consumption and price brackets compared to Residential users (blue).====SEP====### Principal Component Analysis (PCA): Simplifying ComplexityWhen we have 27 columns, it's hard to visualize everything. $PCA$ (Principal Component Analysis) compresses all that information into just two "Super Variables" (PC1 and PC2).- Simple Insight: We managed to capture nearly $50\%$ of the total information in the dataset using just two axes.- What we see: The PCA plot shows a distinct gradient. The color indicates Grid Load %. Since the colors transition smoothly from left to right, we know that the "Super Variables" we created are very good at predicting how stressed the power grid is.====SEP====### Spatial-Temporal Analysis: Geographical PatternsBy combining Region (Space) and Month (Time) into a heatmap, we can see geographical trends.- Simple Insight: Halifax is the undisputed "energy hub" of the dataset, consistently using much more power than regions like Pictou County or the South Shore.- Seasonality: Every region follows the same seasonal cycle—usage is highest in the winter (Months 1-2 and 12) and drops significantly in the spring/fall.====SEP====### Time Series Analysis: The Long-Term TrendWe plotted the consumption from 2015 to 2024 and added a 12-Month Moving Average (the red line).- Simple Insight: While consumption goes up and down every season (the blue spikes), the Red Line is slowly creeping upward over the decade.- Why it matters: This tells us that even after accounting for winter peaks, the overall demand for electricity is growing year-over-year. As a strategist, this means we need to plan for building more capacity for 2025 and beyond.====SEP====## Strategic Summary:1. Halifax First: Since Halifax drives the majority of the load, grid optimization programs should start there for maximum impact.2. Growth Trend: The moving average proves that "Business as Usual" isn't enough; the grid needs to expand to meet the decade-long growth in demand.3. Customer Differentiation: The Pair Plot shows that different customers have different "fingerprints." We should offer tailored energy-saving plans for Residential vs. Commercial users rather than a one-size-fits-all approach.====SEP====1. **Temporal Variables (The Timeline)**These variables anchor our data in time.- Timestamp: The primary key. While currently stored as an object (text), we treat it as Datetime.- Hour, Day, Month, Year: These are Cyclical Numerical values.    - **Note**: Even though they are numbers, we often treat "Hour" as a category because Hour 23 (11 PM) is actually very close to Hour 0 (Midnight), just like Monday is close to Sunday.2. **Numerical Variables (The Measurements)**These are Continuous values where the distance between numbers matters (e.g., the difference between 10°C and 20°C is the same as 20°C to 30°C).- Weather: temperature_c, humidity_pct, wind_speed_kmh, precipitation_mm.- Grid Metrics: consumption_kwh, renewable_pct, grid_load_pct, price_per_kwh.- Environmental Impact: co2_kg.3. **Categorical Variables (The Labels)**These variables represent groups or types. They are Nominal, meaning there is no inherent "order" (Halifax isn't "higher" than Cape Breton; they are just different).- Identity: region, customer_type.- Context: season.- Event Type: anomaly_type (Normal, Spike, Drop, Sustained).4. **Binary/Boolean Flags (The Switches)**These are a special type of categorical data with only two states: 0 (No) or 1 (Yes).- Calendar Events: is_weekend, is_holiday.- Grid Events: power_outage, demand_response, peak_demand_flag.- Quality Control: anomaly_flag.====SEP====## 5. Time-Series Reality CheckFilter the data for the 'Halifax' region and plot a line chart of the first 168 hours (one full week) of the dataset to verify the expected daily 'double-hump' consumption pattern (morning and evening peaks). Save the figure to the output directory.====SEP====## 6. Regional HeatmapsCompare Halifax's consumption load against the other 4 regions (Annapolis Valley, Cape Breton, Pictou County, South Shore).====SEP====### Interpretation: Regional Heatmaps- **Business Insight:** Halifax clearly dominates the consumption landscape, showing significantly higher absolute loads compared to the other four regions year-round. We also observe an intensification of demand during extreme winter and summer months across all regions, particularly in Halifax and Cape Breton.- **Actionable Takeaway:** Grid maintenance and infrastructure upgrades should prioritize the Halifax corridor. Additionally, peak-shaving demand response programs will yield the highest absolute MW reductions if targeted specifically at Halifax during the mid-winter and mid-summer peaks.====SEP====## 7. Seasonal DecompositionSeparate the Halifax dataset into Trend, Seasonality, and Residuals (noise) to understand underlying patterns.====SEP====### Interpretation: Seasonal Decomposition- **Business Insight:** The decomposition isolates the steady baseline growth (Trend) from the predictable yearly cyclical peaks (Seasonality). The 'Residuals' plot highlights unexpected demand spikes (noise) that the regular seasonal model cannot explain.- **Actionable Takeaway:** Our anomaly detection model should focus precisely on these 'Residuals'. By stripping away predictable seasonal weather effects and general growth trends, we can accurately pinpoint true anomalous grid behavior, whether caused by equipment faults, unexpected extreme weather events, or sudden shifts in consumer behavior.====SEP====## 8. Holiday Impact AnalysisAnalyze if `is_holiday` causes a significant drop in commercial demand vs. an increase in residential demand.====SEP====### Interpretation: Holiday Impact Analysis- **Business Insight:** The bar chart effectively illustrates a structural demand shift during statutory holidays. Commercial electricity consumption drops significantly as businesses close, while Residential consumption sees a reciprocal increase due to people staying home.- **Actionable Takeaway:** This confirms `is_holiday` is a critical covariate for our predictive models. The grid must re-route base load expectations on holidays. Anomaly detection models must be aware of this shift; otherwise, the drop in commercial load might trigger a false 'power outage' alert, or the spike in residential load might trigger a false 'surge' alert.====SEP====## 5. Feature EngineeringPreparing the data for advanced modeling (e.g., ARIMA, LSTM). We already converted the timestamp to datetime when loading the dataset.### 5.1 Lag FeaturesCreating lag variables to capture temporal dependencies.====SEP====### 5.2 Weather Interaction FeaturesCreating a 'Heating Degree Day' (HDD) feature to capture the non-linear impact of cold weather on electricity demand.====SEP====### 5.3 Categorical EncodingOne-hot encoding 'region' and 'customer_type' for modeling algorithms that require numerical input.====SEP====